# SciBERT Ordinal Fine-tuning for Paper Quality Classification

This notebook fine-tunes **SciBERT** for an ordinal classification task with labels ranging from **1 to 5**.

Instead of treating the problem as standard multi-class classification, the notebook uses an **ordinal regression formulation**:

- Label 1 → `[0, 0, 0, 0]`
- Label 2 → `[1, 0, 0, 0]`
- Label 3 → `[1, 1, 0, 0]`
- Label 4 → `[1, 1, 1, 0]`
- Label 5 → `[1, 1, 1, 1]`

Main objectives of this notebook:

1. Prepare a Google Colab-compatible environment.
2. Load datasets directly from the GitHub repository.
3. Build textual representations from paper metadata.
4. Fine-tune SciBERT using 5-fold cross-validation.
5. Optimize prediction thresholds using Quadratic Weighted Kappa.
6. Generate final submission files.

The notebook is organized into three major sections:

- Step 1 — Setup
- Step 2 — EDA & Preprocessing
- Step 3 — Training & Inference

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# =========================
# USER CONFIG
# =========================
GITHUB_REPO = "https://github.com/PhatLavar/DATA_MINING_ASSIGNMENT.git"
BRANCH = "main"

PROJECT_NAME = GITHUB_REPO.split("/")[-1].replace(".git", "")

DATA_DIR_NAME = "data"

TRAIN_CSV = "train.csv"
PUBLIC_TEST_CSV = "public_test.csv"
PRIVATE_TEST_CSV = "private_test.csv"

# =========================
# ENV DETECTION
# =========================
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running on Google Colab")

    if not Path(PROJECT_NAME).exists():
        !git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_NAME}
    else:
        print(f"{PROJECT_NAME} already exists")

    %cd {PROJECT_NAME}

else:
    print("Running locally")

ROOT = Path.cwd().resolve()

DATA_DIR = ROOT / DATA_DIR_NAME
MODEL_DIR = ROOT / "models"
SUBMISSION_DIR = ROOT / "submissions"

MODEL_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)

# Step 1 — Dependency Installation

This section checks and installs only the required packages.

Google Colab already includes many common libraries such as:

- numpy
- pandas
- torch
- scikit-learn

Only missing packages are installed dynamically.

Main additional packages:

|     Package    |             Purpose            |
|----------------|--------------------------------|
| `transformers` | HuggingFace Transformer models |
|  `accelerate`  |       Training utilities       |
|     `tqdm`     |         Progress bars          |

In [ ]:
import importlib.util
import subprocess
import sys

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name

    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])
    else:
        print(f"{package_name} already installed")

install_if_missing("numpy")
install_if_missing("pandas")
install_if_missing("scipy")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("transformers")
install_if_missing("accelerate")
install_if_missing("tqdm")

# Global Configuration

This section defines:

- random seeds,
- training hyperparameters,
- optimizer settings,
- Transformer backbone configuration.

Important hyperparameters:

| Parameter | Description |
|---|---|
| `MODEL_NAME` | Pretrained Transformer model |
| `MAX_LENGTH` | Maximum token length |
| `BATCH_SIZE` | Batch size |
| `EPOCHS` | Number of training epochs |
| `BACKBONE_LR` | Learning rate for SciBERT |
| `HEAD_LR` | Learning rate for classification head |

The notebook automatically detects GPU availability.

In [ ]:
import os
import re
import random
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

SEED = 42

MODEL_NAME = "allenai/scibert_scivocab_uncased"

N_SPLITS = 5

MAX_LENGTH = 160

BATCH_SIZE = 8
EPOCHS = 10

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4

WEIGHT_DECAY = 0.01

LABEL_COL = "Label"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

# Step 2 — Dataset Loading

This section loads all CSV files from the `data/` folder.

Expected structure:

```text
project/
├── data/
│   ├── train.csv
│   ├── public_test.csv
│   └── private_test.csv
```

The notebook validates all required files before continuing.

In [ ]:
train_path = DATA_DIR / TRAIN_CSV
public_test_path = DATA_DIR / PUBLIC_TEST_CSV
private_test_path = DATA_DIR / PRIVATE_TEST_CSV

required_files = [
    train_path,
    public_test_path,
    private_test_path
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing file: {file_path}"
        )

train = pd.read_csv(train_path)
public_test = pd.read_csv(public_test_path)
private_test = pd.read_csv(private_test_path)

print("Train shape:", train.shape)
print("Public test shape:", public_test.shape)
print("Private test shape:", private_test.shape)

display(train.head())

# Exploratory Data Analysis (EDA)

This section performs a lightweight exploratory analysis:

- column inspection,
- missing value analysis,
- label distribution analysis,
- sample visualization.

The goal is to verify dataset integrity before preprocessing.

In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nMissing values:")
display(train.isna().sum())

print("\nLabel distribution:")
display(
    train[LABEL_COL]
    .value_counts()
    .sort_index()
)

display(train.head())

# Text Feature Engineering

This section converts structured paper metadata into a single textual sequence.

Features combined:

- title
- venue
- year
- authors
- doi

Example:

```text
Title: xxx [SEP] Venue: xxx [SEP] Year: xxx
```

The notebook also builds:

- venue embeddings,
- venue-to-id mappings,
- unknown venue handling.

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = re.sub(r"\s+", " ", x)

    return x.strip()

def build_input_text(df):

    title = df["title"].map(clean_text)
    venue = df["venue"].map(clean_text)
    year = df["year"].map(clean_text)
    authors = df["authors"].map(clean_text)
    doi = df["doi"].map(clean_text)

    return (
        "Title: " + title +
        " [SEP] Venue: " + venue +
        " [SEP] Year: " + year +
        " [SEP] Authors: " + authors +
        " [SEP] DOI: " + doi
    )

train["input_text"] = build_input_text(train)
public_test["input_text"] = build_input_text(public_test)
private_test["input_text"] = build_input_text(private_test)

venue2id = {"<UNK>": 0}

all_venues = (
    train["venue"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.lower()
    .unique()
)

for venue in sorted(all_venues):

    if venue not in venue2id:
        venue2id[venue] = len(venue2id)

def map_venue(x):

    if pd.isna(x):
        return 0

    return venue2id.get(
        str(x).lower(),
        0
    )

train["venue_id"] = train["venue"].map(map_venue)
public_test["venue_id"] = public_test["venue"].map(map_venue)
private_test["venue_id"] = private_test["venue"].map(map_venue)

NUM_VENUES = len(venue2id)

print("Number of venues:", NUM_VENUES)

display(
    train[
        ["input_text", "venue_id", LABEL_COL]
    ].head()
)

# Ordinal Regression Utilities

This notebook uses ordinal regression instead of standard classification.

Advantages:

- preserves label ordering,
- improves ranking consistency,
- often improves QWK performance.

Main utilities:

| Function | Purpose |
|---|---|
| `labels_to_ordinal_targets()` | Converts labels into ordinal vectors |
| `apply_thresholds()` | Converts continuous scores into labels |
| `optimize_thresholds()` | Maximizes Quadratic Weighted Kappa |

In [ ]:
def quadratic_weighted_kappa(y_true, y_pred):

    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

def labels_to_ordinal_targets(labels):

    labels = np.asarray(labels).astype(int)

    targets = np.zeros(
        (len(labels), 4),
        dtype=np.float32
    )

    for i, label in enumerate(labels):
        targets[i, :label - 1] = 1.0

    return targets

def ordinal_logits_to_continuous(logits):
    probs = torch.sigmoid(logits)
    return 1.0 + probs.sum(dim=1)

def apply_thresholds(pred, thresholds):
    return np.digitize(pred, thresholds) + 1

def optimize_thresholds(y_true, pred, initial=None):
    if initial is None:
        initial = np.array([1.5, 2.5, 3.5, 4.5], dtype=float)

    def loss(thresholds):
        thresholds = np.sort(thresholds)
        y_hat = apply_thresholds(pred, thresholds)
        return -quadratic_weighted_kappa(y_true, y_hat)

    result = minimize(
        loss,
        x0=initial,
        method="Nelder-Mead",
        options={
            "maxiter": 2000,
            "xatol": 1e-6,
            "fatol": 1e-6,
        },
    )

    thresholds = np.sort(result.x)
    score = -result.fun

    return thresholds, score

# Step 3 — Training

This section contains:

1. Dataset pipeline
2. Tokenization
3. Model architecture
4. Training loop
5. Cross-validation
6. Threshold optimization
7. Inference & submission generation

Training uses:

- 5-fold Stratified Cross Validation,
- BCEWithLogitsLoss,
- AdamW optimizer,
- SciBERT backbone,
- Multi-sample dropout,
- Quadratic Weighted Kappa evaluation.

# Dataset Pipeline & Tokenization

This section defines the dataset pipeline used during training.

Main responsibilities:

- tokenize text using SciBERT tokenizer,
- pad sequences,
- truncate long inputs,
- prepare ordinal regression targets,
- prepare venue metadata.

The tokenizer used:

```python
AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
```

Maximum sequence length:

```python
MAX_LENGTH = 160
```

A relatively small max length is used to reduce GPU memory usage while still preserving most important metadata information.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PaperDataset(Dataset):

    def __init__(
        self,
        df,
        labels=None,
        max_length=160
    ):

        self.texts = df["input_text"].values

        self.venue_ids = (
            df["venue_id"]
            .values
            .astype(np.int64)
        )

        self.labels = labels
        self.max_length = max_length

        if labels is not None:
            self.ordinal_targets = (
                labels_to_ordinal_targets(labels)
            )
        else:
            self.ordinal_targets = None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoded = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids":
                encoded["input_ids"].squeeze(0),

            "attention_mask":
                encoded["attention_mask"].squeeze(0),

            "venue_id":
                torch.tensor(
                    self.venue_ids[idx],
                    dtype=torch.long
                )
        }

        if self.labels is not None:

            item["ordinal_target"] = torch.tensor(
                self.ordinal_targets[idx],
                dtype=torch.float
            )

            item["label"] = torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )

        return item

# SciBERT Ordinal Architecture

The model architecture combines:

1. SciBERT Transformer backbone
2. Mean pooling
3. Max pooling
4. Venue embeddings
5. Multi-sample dropout
6. Ordinal regression head

Key architectural decisions:

| Component | Purpose |
|---|---|
| Mean pooling | Captures overall contextual meaning |
| Max pooling | Captures strong local signals |
| Venue embedding | Injects structured metadata |
| Multi-sample dropout | Improves generalization |
| Partial layer unfreezing | Reduces overfitting |

Instead of fully fine-tuning the entire Transformer, only the last few layers are unfrozen.
This stabilizes training and reduces GPU memory consumption.

In [ ]:
class OrdinalSciBERTModel(nn.Module):

    def __init__(
        self,
        model_name,
        num_venues,
        venue_dim=32,
        dropout=0.2,
        n_dropout_samples=4,
        unfreeze_last_n_layers=2
    ):

        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_name)

        hidden_size = self.backbone.config.hidden_size

        # Freeze all layers first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze last transformer layers
        if hasattr(self.backbone, "encoder"):

            total_layers = len(
                self.backbone.encoder.layer
            )

            for layer_idx in range(
                total_layers - unfreeze_last_n_layers,
                total_layers
            ):

                for param in (
                    self.backbone
                    .encoder
                    .layer[layer_idx]
                    .parameters()
                ):
                    param.requires_grad = True

        # Unfreeze pooler
        if (
            hasattr(self.backbone, "pooler")
            and self.backbone.pooler is not None
        ):

            for param in (
                self.backbone.pooler.parameters()
            ):
                param.requires_grad = True

        # Venue embedding
        self.venue_embedding = nn.Embedding(
            num_venues,
            venue_dim
        )

        feature_dim = (
            hidden_size * 2 + venue_dim
        )

        # Multi-sample dropout
        self.dropouts = nn.ModuleList([
            nn.Dropout(dropout)
            for _ in range(n_dropout_samples)
        ])

        # Ordinal regression head
        self.head = nn.Sequential(

            nn.Linear(feature_dim, 256),

            nn.ReLU(),

            nn.LayerNorm(256),

            nn.Dropout(dropout),

            nn.Linear(256, 4)
        )

    def mean_max_pooling(
        self,
        last_hidden_state,
        attention_mask
    ):

        mask = attention_mask.unsqueeze(-1).float()

        mean_pool = (
            (last_hidden_state * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1e-9)
        )

        masked_hidden = last_hidden_state.masked_fill(
            mask == 0,
            -1e9
        )

        max_pool = masked_hidden.max(dim=1).values

        return torch.cat(
            [mean_pool, max_pool],
            dim=1
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        venue_id
    ):

        output = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = self.mean_max_pooling(
            output.last_hidden_state,
            attention_mask
        )

        venue_emb = self.venue_embedding(venue_id)

        features = torch.cat(
            [pooled, venue_emb],
            dim=1
        )

        logits = 0

        for dropout in self.dropouts:
            logits += self.head(
                dropout(features)
            )

        logits /= len(self.dropouts)

        return logits

# Training Utilities

This section defines helper functions used during training and validation.

Main components:

| Function | Purpose |
|---|---|
| `train_one_epoch()` | Single training epoch |
| `predict_continuous()` | Continuous ordinal prediction |
| `build_optimizer()` | Optimizer initialization |

Training details:

- BCEWithLogitsLoss is used for ordinal targets.
- Gradient clipping stabilizes training.
- AdamW optimizer is used.
- Linear warmup scheduling is applied.

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    criterion
):

    model.train()

    total_loss = 0.0

    for batch in tqdm(loader, leave=False):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        venue_id = batch["venue_id"].to(device)

        ordinal_target = (
            batch["ordinal_target"].to(device)
        )

        logits = model(
            input_ids,
            attention_mask,
            venue_id
        )

        loss = criterion(
            logits,
            ordinal_target
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += (
            loss.item() * input_ids.size(0)
        )

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_continuous(model, loader):

    model.eval()

    preds = []

    for batch in tqdm(loader, leave=False):

        input_ids = batch["input_ids"].to(device)

        attention_mask = (
            batch["attention_mask"].to(device)
        )

        venue_id = batch["venue_id"].to(device)

        logits = model(
            input_ids,
            attention_mask,
            venue_id
        )

        continuous = (
            ordinal_logits_to_continuous(logits)
        )

        preds.append(
            continuous.cpu().numpy()
        )

    return np.concatenate(preds)


def build_optimizer(model):

    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if name.startswith("backbone"):
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": backbone_params,
                "lr": BACKBONE_LR
            },
            {
                "params": head_params,
                "lr": HEAD_LR
            }
        ],
        weight_decay=WEIGHT_DECAY
    )

    return optimizer

# 5-Fold Cross Validation Training

This section performs full model training using Stratified K-Fold cross validation.

Workflow:

1. Split dataset into folds
2. Train one model per fold
3. Save best fold checkpoint
4. Generate OOF predictions
5. Optimize thresholds
6. Compute final QWK score

Advantages of cross validation:

- more robust evaluation,
- reduced overfitting,
- better ensemble performance,
- more stable leaderboard scores.

In [ ]:
y = train[LABEL_COL].astype(int).values

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

oof_pred = np.zeros(
    len(train),
    dtype=np.float32
)

fold_thresholds = []
fold_scores = []
model_paths = []

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(train, y),
    1
):

    print(
        f"\n========== Fold {fold}/{N_SPLITS} =========="
    )

    train_df = (
        train.iloc[tr_idx]
        .reset_index(drop=True)
    )

    valid_df = (
        train.iloc[va_idx]
        .reset_index(drop=True)
    )

    y_train = y[tr_idx]
    y_valid = y[va_idx]

    train_dataset = PaperDataset(
        train_df,
        labels=y_train,
        max_length=MAX_LENGTH
    )

    valid_dataset = PaperDataset(
        valid_df,
        labels=y_valid,
        max_length=MAX_LENGTH
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    model = OrdinalSciBERTModel(
        model_name=MODEL_NAME,
        num_venues=NUM_VENUES,
        venue_dim=32,
        dropout=0.25,
        n_dropout_samples=4,
        unfreeze_last_n_layers=2
    ).to(device)

    optimizer = build_optimizer(model)

    total_steps = len(train_loader) * EPOCHS

    warmup_steps = int(total_steps * 0.1)

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
    )

    criterion = nn.BCEWithLogitsLoss()

    best_qwk = -1

    best_thresholds = None

    best_model_path = (
        MODEL_DIR / f"fold_{fold}.pt"
    )

    for epoch in range(1, EPOCHS + 1):

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            criterion
        )

        valid_pred = predict_continuous(
            model,
            valid_loader
        )

        thresholds, valid_qwk = (
            optimize_thresholds(
                y_valid,
                valid_pred
            )
        )

        print(
            f"Fold {fold} | "
            f"Epoch {epoch} | "
            f"Loss: {train_loss:.5f} | "
            f"QWK: {valid_qwk:.6f}"
        )

        if valid_qwk > best_qwk:

            best_qwk = valid_qwk

            best_thresholds = thresholds

            torch.save(
                model.state_dict(),
                best_model_path
            )

    print(
        f"Best Fold QWK: {best_qwk:.6f}"
    )

    model.load_state_dict(
        torch.load(
            best_model_path,
            map_location=device
        )
    )

    model.to(device)

    best_valid_pred = predict_continuous(
        model,
        valid_loader
    )

    oof_pred[va_idx] = best_valid_pred

    fold_thresholds.append(
        best_thresholds
    )

    fold_scores.append(best_qwk)

    model_paths.append(best_model_path)

    del model

    torch.cuda.empty_cache()

# Threshold Optimization & OOF Evaluation

After cross-validation training, the notebook evaluates Out-Of-Fold (OOF) predictions.

OOF predictions are important because they:

- simulate unseen validation performance,
- provide reliable QWK estimation,
- reduce leaderboard overfitting risk.

The notebook then:

1. optimizes global thresholds,
2. converts continuous predictions into labels,
3. computes final OOF Quadratic Weighted Kappa.

In [ ]:
oof_thresholds, oof_qwk = optimize_thresholds(
    y,
    oof_pred
)

oof_labels = apply_thresholds(
    oof_pred,
    oof_thresholds
)

print("\n========== OOF Evaluation ==========")

print("Fold Scores:")
print(fold_scores)

print("\nMean Fold QWK:")
print(np.mean(fold_scores))

print("\nOOF QWK:")
print(oof_qwk)

print("\nOptimized Thresholds:")
print(oof_thresholds)

print("\nPrediction Distribution:")
print(
    pd.Series(oof_labels)
    .value_counts()
    .sort_index()
)

# Save OOF Predictions

This section saves Out-Of-Fold predictions for future analysis.

The saved file contains:

- ground truth labels,
- continuous predictions,
- final thresholded labels.

This file is useful for:

- error analysis,
- ensemble experiments,
- threshold tuning,
- debugging.

In [ ]:
oof_df = pd.DataFrame({
    "id": train["id"].values,
    "y_true": y,
    "pred_continuous": oof_pred,
    "pred_label": oof_labels
})

oof_path = (
    SUBMISSION_DIR /
    "oof_predictions.csv"
)

oof_df.to_csv(
    oof_path,
    index=False
)

print("Saved OOF predictions:")
print(oof_path)

display(oof_df.head())

# Ensemble Inference

This section performs inference on:

- public test set,
- private test set.

Inference workflow:

1. Load each fold checkpoint
2. Predict continuous ordinal scores
3. Average fold predictions
4. Apply optimized thresholds
5. Generate final labels

Using fold ensembling usually improves:

- stability,
- robustness,
- leaderboard performance.

In [ ]:
public_dataset = PaperDataset(
    public_test,
    labels=None,
    max_length=MAX_LENGTH
)

private_dataset = PaperDataset(
    private_test,
    labels=None,
    max_length=MAX_LENGTH
)

public_loader = DataLoader(
    public_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0
)

private_loader = DataLoader(
    private_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0
)

public_pred = np.zeros(
    len(public_test),
    dtype=np.float32
)

private_pred = np.zeros(
    len(private_test),
    dtype=np.float32
)

for fold, model_path in enumerate(
    model_paths,
    1
):

    print(
        f"Inference with fold {fold}"
    )

    model = OrdinalSciBERTModel(
        model_name=MODEL_NAME,
        num_venues=NUM_VENUES,
        venue_dim=32,
        dropout=0.25,
        n_dropout_samples=4,
        unfreeze_last_n_layers=2
    ).to(device)

    model.load_state_dict(
        torch.load(
            model_path,
            map_location=device
        )
    )

    model.to(device)

    public_pred += (
        predict_continuous(
            model,
            public_loader
        ) / len(model_paths)
    )

    private_pred += (
        predict_continuous(
            model,
            private_loader
        ) / len(model_paths)
    )

    del model

    torch.cuda.empty_cache()

# Submission Generation

This section converts continuous predictions into final integer labels.

The notebook:

1. applies optimized thresholds,
2. builds submission DataFrames,
3. concatenates public and private predictions,
4. exports final CSV files.

Final outputs are saved inside:

```text
submissions/
```

Main generated files:

| File | Description |
|---|---|
| `oof_predictions.csv` | OOF validation predictions |
| `submission.csv` | Final test predictions |

In [ ]:
public_labels = apply_thresholds(
    public_pred,
    oof_thresholds
)

private_labels = apply_thresholds(
    private_pred,
    oof_thresholds
)

public_submission = pd.DataFrame({
    "id": public_test["id"].values,
    "Label": public_labels.astype(int)
})

private_submission = pd.DataFrame({
    "id": private_test["id"].values,
    "Label": private_labels.astype(int)
})

submission = pd.concat(
    [
        public_submission,
        private_submission
    ],
    axis=0,
    ignore_index=True
)

submission_path = (
    SUBMISSION_DIR /
    "submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print("Submission saved:")
print(submission_path)

print("\nSubmission shape:")
print(submission.shape)

print("\nPrediction distribution:")
print(
    submission["Label"]
    .value_counts()
    .sort_index()
)

display(submission.head())

# Final Notes

This notebook is designed to be:

- fully runnable on Google Colab,
- reproducible,
- modular,
- easy to modify and extend.

Possible future improvements:

- pseudo-labeling,
- larger Transformer models,
- advanced pooling strategies,
- adversarial training,
- mixed precision training,
- weighted ensemble methods.

Recommended next steps:

1. Save trained checkpoints to Google Drive.
2. Track experiments systematically.
3. Compare QWK across different architectures.
4. Perform detailed error analysis using OOF predictions.